# Phase 2 Benchmark Runner (Colab)
Thin orchestration notebook. Training logic stays in `src/`.


In [3]:
# Colab setup + repo bootstrap
import os
from google.colab import drive
drive.mount('/content/drive')

!unzip -q -n "/content/drive/MyDrive/PlantVillage/Datasets.zip" -d "/content/"


if not os.path.isdir('/content/Plant-Disease-Detection-CV'):
    !git clone https://github.com/WilliamKyaww/Plant-Disease-Detection-CV.git
else:
    !cd /content/Plant-Disease-Detection-CV && git pull

!pip install -q -r /content/Plant-Disease-Detection-CV/requirements.txt

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Already up to date.


In [4]:
# Canonical repo root
from pathlib import Path
import os, sys

repo_root = Path('/content/Plant-Disease-Detection-CV')  # cloned repo path
if not (repo_root / 'src').is_dir():
    raise FileNotFoundError(f"Repo not found at {repo_root}. Run clone cell first.")

os.chdir(repo_root)
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

print('Using repo root:', repo_root)
print('Executed in Colab:', os.path.isdir('/content'))

Using repo root: /content/Plant-Disease-Detection-CV
Executed in Colab: True


In [5]:
# Frozen split guard + dry-run benchmark
from pathlib import Path
manifest = Path('results/split_manifests/latest_split_manifest.json')
if manifest.exists():
    print('Using existing frozen splits (no regeneration).')
    print('Manifest:', manifest.resolve())
else:
    !python -m src.prepare_splits

!python -m src.run_phase2_benchmark --dry-run --no-pretrained --batch-size 4 --num-workers 0 --amp


Using existing frozen splits (no regeneration).
Manifest: /content/Plant-Disease-Detection-CV/results/split_manifests/latest_split_manifest.json
AMP requested: enabled for CUDA training.
Phase 2 configuration
------------------------------------------------------------
Models: ['resnet18', 'resnet50', 'efficientnet_b0', 'vit_small_patch16_224']
Seeds: [41, 42, 43]
Epochs: 30
Batch size: 4
Class weighting: inverse_frequency
Scheduler: cosine
Pretrained: False
Dry-run: True
Out dir: /content/Plant-Disease-Detection-CV/results/phase2
Dry-run report saved: /content/Plant-Disease-Detection-CV/results/phase2/phase2_dry_run_report.json


In [6]:
# Minimal training smoke (1 model, 1 seed, 1 epoch)
!python -m src.run_phase2_benchmark --models resnet18 --seeds 41 --epochs 1 --batch-size 32 --num-workers 2 --scheduler cosine --class-weighting inverse_frequency


Phase 2 configuration
------------------------------------------------------------
Models: ['resnet18']
Seeds: [41]
Epochs: 1
Batch size: 32
Class weighting: inverse_frequency
Scheduler: cosine
Pretrained: True
Dry-run: False
Out dir: /content/Plant-Disease-Detection-CV/results/phase2
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth
100% 44.7M/44.7M [00:00<00:00, 129MB/s]

Epoch 1/1
----------------------------------------
  train  Loss: 0.3163  Acc: 0.9034
    val  Loss: 0.1500  Acc: 0.9516

Training complete in 1.3 min
Best val accuracy: 0.9516
Accuracy: 0.9470  (2932/3096)
Model saved to /content/Plant-Disease-Detection-CV/models/phase2/resnet18/seed_41/best.pth
Experiment log saved: /content/Plant-Disease-Detection-CV/results/phase2/runs/resnet18/seed_41/phase2_resnet18_seed41.json
Saved: /content/Plant-Disease-Detection-CV/results/phase2/model_seed_metrics.csv
Saved: /content/Plant-Disease-Detectio

In [7]:
# Package key artifacts and download as one zip
from pathlib import Path
import zipfile
from google.colab import files

artifact_paths = [
    "results/phase2/model_seed_metrics.csv",
    "results/phase2/model_summary_mean_std.csv",
    "results/phase2/leaderboard.csv",
    "results/phase2/phase2_dry_run_report.json",
    "results/phase2/runs/resnet18/seed_41/metrics.json",
    "results/phase2/runs/resnet18/seed_41/phase2_resnet18_seed41.json",
    "results/colab_smoke/latest_colab_smoke.json",
    "results/colab_smoke/latest_colab_smoke.txt",
]

zip_name = "phase2_artifacts_export.zip"
with zipfile.ZipFile(zip_name, "w", zipfile.ZIP_DEFLATED) as zf:
    for p in artifact_paths:
        path = Path(p)
        if path.exists():
            zf.write(path, arcname=str(path))
        else:
            print(f"Missing (skipped): {p}")

files.download(zip_name)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [8]:
# Full benchmark run
!python -m src.run_phase2_benchmark --models resnet18,resnet50,efficientnet_b0,vit_small_patch16_224 --seeds 41,42,43 --epochs 30 --batch-size 32 --num-workers 2 --scheduler cosine --class-weighting inverse_frequency --pretrained --amp --resume

AMP requested: enabled for CUDA training.
Phase 2 configuration
------------------------------------------------------------
Models: ['resnet18', 'resnet50', 'efficientnet_b0', 'vit_small_patch16_224']
Seeds: [41, 42, 43]
Epochs: 30
Batch size: 32
Class weighting: inverse_frequency
Scheduler: cosine
Pretrained: True
Dry-run: False
Out dir: /content/Plant-Disease-Detection-CV/results/phase2
Skipping existing run: resnet18 seed 41

Epoch 1/30
----------------------------------------
  train  Loss: 0.3091  Acc: 0.9041
    val  Loss: 0.3143  Acc: 0.9070

Epoch 2/30
----------------------------------------
  train  Loss: 0.1555  Acc: 0.9512
    val  Loss: 0.5385  Acc: 0.8117

Epoch 3/30
----------------------------------------
  train  Loss: 0.0999  Acc: 0.9659
    val  Loss: 0.0867  Acc: 0.9742

Epoch 4/30
----------------------------------------
  train  Loss: 0.0848  Acc: 0.9721
    val  Loss: 0.0599  Acc: 0.9764

Epoch 5/30
----------------------------------------
  train  Loss: 0.0725 

In [13]:
# Export to a zip file and download artifacts
from pathlib import Path
from zipfile import ZipFile, ZIP_DEFLATED
from google.colab import files

REPO = Path("/content/Plant-Disease-Detection-CV")
OUT_ZIP = Path("/content/phase2_artifacts_export.zip")

# Keep this False unless you explicitly want all model checkpoints too (much larger zip).
INCLUDE_MODELS = False

paths_to_pack = [
    REPO / "results/phase2/model_seed_metrics.csv",
    REPO / "results/phase2/model_summary_mean_std.csv",
    REPO / "results/phase2/leaderboard.csv",
    REPO / "results/phase2/runs",
    REPO / "results/split_manifests/latest_split_manifest.json",
]

if INCLUDE_MODELS:
    paths_to_pack.append(REPO / "models/phase2")

with ZipFile(OUT_ZIP, "w", ZIP_DEFLATED) as zf:
    for p in paths_to_pack:
        if not p.exists():
            print(f"[skip] missing: {p}")
            continue
        if p.is_file():
            zf.write(p, arcname=p.relative_to(REPO))
        else:
            for f in p.rglob("*"):
                if f.is_file():
                    zf.write(f, arcname=f.relative_to(REPO))

print(f"Created: {OUT_ZIP} ({OUT_ZIP.stat().st_size / (1024**2):.2f} MB)")
files.download(str(OUT_ZIP))


Created: /content/phase2_artifacts_export.zip (0.32 MB)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>